# Prediksi Tingkat Emisi Karbon Dioksida ($CO_2$) Kendaraan Bermotor Menggunakan Multiple Linear Regression dan Random Forest Regressor

**Identitas Mahasiswa:**
- **Nama:** Sigit Miraj Permana
- **NIM:** 301240037
- **Kelas:** 5B — Teknik Informatika (FTI UNIBBA)
- **Mata Kuliah:** Pembelajaran Mesin (UTS Semester 5)
- **Sumber Dataset:** *CO2 Emission by Vehicles* (Kaggle / Natural Resources Canada)


## 1. Import Library & Konfigurasi Lingkungan


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
sns.set_style('whitegrid')
print("Library siap digunakan.")


## 2. Pemuatan & Audit Struktur Dataset


In [ ]:
# Unggah file 'CO2 Emissions_Canada.csv' ke Google Colab atau baca dari path lokal
df_raw = pd.read_csv('CO2 Emissions_Canada.csv')
print(f"Dimensi Dataset Mentah: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")
display(df_raw.head())
print("\nJumlah Missing Values per Kolom:")
print(df_raw.isnull().sum())
print(f"\nJumlah Baris Duplikat Identik: {df_raw.duplicated().sum()}")


## 3. Pra-Pemrosesan Data (Deduplikasi & Penanganan Anomali Fuel Type)


In [ ]:
# Menghapus 1.103 baris duplikat identik untuk mencegah data leakage
df_clean = df_raw.drop_duplicates().copy()

# Menghapus kelas minoritas ekstrem Fuel Type = 'N' (hanya 1 sampel)
print("Distribusi Fuel Type sebelum filtering:\n", df_clean['Fuel Type'].value_counts())
df_clean = df_clean[df_clean['Fuel Type'] != 'N'].reset_index(drop=True)
print(f"\nDimensi Dataset Bersih Akhir: {df_clean.shape[0]} baris x {df_clean.shape[1]} kolom")


## 4. Exploratory Data Analysis (EDA) & Visualisasi Tema Putih Standar Jurnal


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

sns.histplot(df_clean['CO2 Emissions(g/km)'], bins=35, kde=True, color='#2563EB', ax=axes[0])
axes[0].axvline(df_clean['CO2 Emissions(g/km)'].mean(), color='#DC2626', linestyle='--', label='Mean')
axes[0].set_title('Distribusi Target Emisi CO2 (g/km)')
axes[0].legend()

num_cols = ['Engine Size(L)', 'Cylinders', 'Fuel Consumption City (L/100 km)',
            'Fuel Consumption Hwy (L/100 km)', 'Fuel Consumption Comb (L/100 km)', 'CO2 Emissions(g/km)']
sns.heatmap(df_clean[num_cols].corr(), annot=True, fmt='.2f', cmap='Blues', ax=axes[1])
axes[1].set_title('Matriks Korelasi Pearson Fitur Numerik')
plt.tight_layout()
plt.show()


## 5. Pembagian Data (Train-Test Split 80:20) & Pelatihan Model


In [ ]:
feature_cols = [
    'Vehicle Class', 'Engine Size(L)', 'Cylinders', 'Transmission', 'Fuel Type',
    'Fuel Consumption City (L/100 km)', 'Fuel Consumption Hwy (L/100 km)',
    'Fuel Consumption Comb (L/100 km)'
]
cat_features = ['Vehicle Class', 'Transmission', 'Fuel Type']
num_features = [
    'Engine Size(L)', 'Cylinders',
    'Fuel Consumption City (L/100 km)', 'Fuel Consumption Hwy (L/100 km)',
    'Fuel Consumption Comb (L/100 km)'
]

X = df_clean[feature_cols]
y = df_clean['CO2 Emissions(g/km)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('num', 'passthrough', num_features)
])

# 1. Multiple Linear Regression
mlr_model = Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])
mlr_model.fit(X_train, y_train)
y_pred_mlr = mlr_model.predict(X_test)

# 2. Random Forest Regressor
rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=60, max_depth=16, random_state=42, n_jobs=-1))
])
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

def evaluate(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f"{name:30s} | R2: {r2:.5f} | MAE: {mae:.3f} g/km | RMSE: {rmse:.3f} g/km")

print("=== HASIL EVALUASI DATA UJI (20%) ===")
evaluate("Multiple Linear Regression", y_test, y_pred_mlr)
evaluate("Random Forest Regressor", y_test, y_pred_rf)


## 6. Validasi Silang (5-Fold Cross-Validation) & Ekspor Model


In [ ]:
cv_mlr = cross_val_score(mlr_model, X, y, cv=5, scoring='r2')
cv_rf = cross_val_score(rf_model, X, y, cv=5, scoring='r2')
print(f"5-Fold CV R2 MLR: {cv_mlr.mean():.5f} (+/- {cv_mlr.std():.5f})")
print(f"5-Fold CV R2 RF : {cv_rf.mean():.5f} (+/- {cv_rf.std():.5f})")

joblib.dump(mlr_model, 'mlr_co2_model.joblib', compress=3)
joblib.dump(rf_model, 'rf_co2_model.joblib', compress=3)
print("Model berhasil diekspor untuk deployment aplikasi web Flask.")
